<a href="https://colab.research.google.com/github/Amt2525/classificador_de_arquivos/blob/main/analise_imagens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# BLOCO 02 — Reinstalar o OpenCV

%pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless opencv-contrib-python-headless
%pip install opencv-python-headless


Found existing installation: opencv-python-headless 5.0.0.93
Uninstalling opencv-python-headless-5.0.0.93:
  Successfully uninstalled opencv-python-headless-5.0.0.93
  Using cached opencv_python_headless-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
Using cached opencv_python_headless-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl (61.2 MB)


In [ ]:

# BLOCO 03 — Testar o detector de rostos

import cv2

DETECTOR_ROSTO = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

if DETECTOR_ROSTO.empty():
    print("Erro: não foi possível carregar o detector.")
else:
    print("Detector de rostos carregado com sucesso!")


AttributeError: module 'cv2' has no attribute 'CascadeClassifier'

In [ ]:

# BLOCO 04 — Identificar o módulo cv2

import cv2

print("Arquivo carregado:", cv2.__file__)
print("Versão:", getattr(cv2, "__version__", "Não identificada"))
print("CascadeClassifier existe:", hasattr(cv2, "CascadeClassifier"))
print("Conteúdo do módulo:", [x for x in dir(cv2) if "Cascade" in x or x == "data"])


Arquivo carregado: /usr/local/lib/python3.13/dist-packages/cv2/__init__.py
Versão: 5.0.0
CascadeClassifier existe: False
Conteúdo do módulo: ['data']


In [ ]:

# BLOCO 05 — Instalar OpenCV 4

%pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless opencv-contrib-python-headless
%pip install opencv-contrib-python-headless==4.11.0.86


Found existing installation: opencv-python-headless 5.0.0.93
Uninstalling opencv-python-headless-5.0.0.93:
  Successfully uninstalled opencv-python-headless-5.0.0.93
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 MB 46.4 MB/s  0:00:01


In [1]:
import os
import re
import time
import unicodedata

import cv2
import pandas as pd
import pytesseract
from pytesseract import Output


# 1. CONFIGURACAO DAS PASTAS
PASTA_BASE = "/content/arquivos"

PASTAS = {
    "rg_antigo": os.path.join(PASTA_BASE, "rg_antigos"),
    "rg_novo": os.path.join(PASTA_BASE, "rg_novos"),
    "invalido": os.path.join(PASTA_BASE, "invalidos"),
}

ARQUIVO_CSV = "/content/analise_documentos.csv"

EXTENSOES_IMAGEM = (
    ".jpg", ".jpeg", ".png", ".bmp",
    ".webp", ".tif", ".tiff"
)


# 2. CARREGAR DETECTORES UMA UNICA VEZ
DETECTOR_ROSTO = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

DETECTOR_QR = cv2.QRCodeDetector()

PALAVRAS_CHAVE = [
    "BRASIL", "REPUBLICA", "IDENTIFICACAO", "CPF",
    "NASCIMENTO", "FILIACAO", "REGISTRO GERAL",
    "CARTEIRA DE IDENTIDADE",
]


# 3. NORMALIZAR TEXTO
def normalizar_texto(texto):
    texto = texto.upper()
    texto = unicodedata.normalize("NFD", texto)

    return "".join(
        caractere for caractere in texto
        if unicodedata.category(caractere) != "Mn"
    )


# 4. LER O TEXTO E ESCOLHER A MELHOR ROTACAO
def ler_melhor_texto(imagem):
    orientacoes = {
        "original": imagem,
        "90_direita": cv2.rotate(
            imagem, cv2.ROTATE_90_CLOCKWISE
        ),
        "90_esquerda": cv2.rotate(
            imagem, cv2.ROTATE_90_COUNTERCLOCKWISE
        ),
        "180": cv2.rotate(
            imagem, cv2.ROTATE_180
        ),
    }

    melhor_texto = ""
    melhor_orientacao = "original"
    melhor_pontuacao = -1
    melhor_confianca = -1

    for nome, imagem_rotacionada in orientacoes.items():
        cinza = cv2.cvtColor(
            imagem_rotacionada,
            cv2.COLOR_BGR2GRAY
        )

        # Extrai texto e confianca em uma unica execucao
        dados = pytesseract.image_to_data(
            cinza,
            lang="por",
            config="--psm 6",
            output_type=Output.DICT,
        )

        palavras = []
        confiancas = []

        for palavra, valor in zip(
            dados["text"], dados["conf"]
        ):
            palavra = palavra.strip()

            if palavra:
                palavras.append(palavra)

            try:
                confianca_palavra = float(valor)

                if confianca_palavra >= 0:
                    confiancas.append(confianca_palavra)

            except (ValueError, TypeError):
                pass

        texto = " ".join(palavras)
        texto_normalizado = normalizar_texto(texto)

        pontuacao = sum(
            chave in texto_normalizado
            for chave in PALAVRAS_CHAVE
        )

        confianca = (
            sum(confiancas) / len(confiancas)
            if confiancas else 0
        )

        if (
            pontuacao > melhor_pontuacao
            or (
                pontuacao == melhor_pontuacao
                and confianca > melhor_confianca
            )
        ):
            melhor_texto = texto
            melhor_orientacao = nome
            melhor_pontuacao = pontuacao
            melhor_confianca = confianca

        # Interrompe quando encontra varias palavras-chave
        if pontuacao >= 3:
            break

    return (
        melhor_texto,
        melhor_orientacao,
        melhor_pontuacao,
        melhor_confianca,
    )


# 5. ANALISAR UMA IMAGEM
def analisar_imagem(caminho, classe):
    imagem = cv2.imread(caminho)

    if imagem is None:
        print("Nao foi possivel abrir:", caminho)
        return None

    altura, largura = imagem.shape[:2]

    proporcao = (
        round(largura / altura, 3)
        if altura else 0
    )

    (
        texto,
        orientacao,
        pontuacao,
        confianca,
    ) = ler_melhor_texto(imagem)

    texto_normalizado = normalizar_texto(texto)

    # Caracteristicas encontradas no texto
    tem_rg = int(
        "REGISTRO GERAL" in texto_normalizado
        or re.search(
            r"\bRG\b", texto_normalizado
        ) is not None
    )

    tem_cpf = int("CPF" in texto_normalizado)
    tem_brasil = int("BRASIL" in texto_normalizado)
    tem_republica = int("REPUBLICA" in texto_normalizado)

    tem_identidade = int(
        "CARTEIRA DE IDENTIDADE" in texto_normalizado
        or "IDENTIDADE NACIONAL" in texto_normalizado
        or "IDENTIFICACAO" in texto_normalizado
    )

    tem_nascimento = int(
        "NASCIMENTO" in texto_normalizado
        or "DATA DE NASC" in texto_normalizado
    )

    tem_filiacao = int(
        "FILIACAO" in texto_normalizado
    )

    tem_filiacao_bilingue = int(
        tem_filiacao and (
            "PARENTAGE" in texto_normalizado
            or "FILIATION" in texto_normalizado
        )
    )

    tem_cnh = int(
        "CARTEIRA NACIONAL DE HABILITACAO"
        in texto_normalizado
        or re.search(
            r"\bCNH\b", texto_normalizado
        ) is not None
    )

    palavras = texto.split()

    # Detectar QR Code
    try:
        pontos_qr = DETECTOR_QR.detect(imagem)[1]
        tem_qr = int(pontos_qr is not None)

    except cv2.error:
        tem_qr = 0

    # Detectar rosto
    try:
        cinza = cv2.cvtColor(
            imagem, cv2.COLOR_BGR2GRAY
        )

        rostos = DETECTOR_ROSTO.detectMultiScale(
            cinza,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(25, 25),
        )

        tem_rosto = int(len(rostos) > 0)

    except cv2.error:
        tem_rosto = None

    # Calcular nitidez
    cinza = cv2.cvtColor(
        imagem, cv2.COLOR_BGR2GRAY
    )

    nitidez = round(
        cv2.Laplacian(
            cinza, cv2.CV_64F
        ).var(),
        2
    )

    return {
        "arquivo": os.path.basename(caminho),
        "classe": classe,
        "largura_px": largura,
        "altura_px": altura,
        "proporcao_largura_altura": proporcao,
        "orientacao_ocr_escolhida": orientacao,
        "pontuacao_palavras_chave": pontuacao,
        "confianca_media_ocr": round(confianca, 2),
        "quantidade_caracteres": len(texto_normalizado),
        "quantidade_palavras": len(palavras),
        "texto_extraido": texto.strip(),
        "tem_texto_rg": tem_rg,
        "tem_cpf": tem_cpf,
        "tem_republica": tem_republica,
        "tem_brasil": tem_brasil,
        "tem_carteira_de_identidade": tem_identidade,
        "tem_data_nascimento": tem_nascimento,
        "tem_filiacao": tem_filiacao,
        "tem_filiacao_bilingue": tem_filiacao_bilingue,
        "possivel_cnh": tem_cnh,
        "tem_qr_code": tem_qr,
        "tem_rosto_detectado": tem_rosto,
        "nitidez_imagem": nitidez,
    }


# 6. ANALISAR TODAS AS IMAGENS
def executar_analise():
    inicio = time.time()

    resultados = []
    erros = []

    for classe, pasta in PASTAS.items():
        if not os.path.isdir(pasta):
            raise FileNotFoundError(
                f"Pasta nao encontrada: {pasta}. "
                "Confira a configuracao."
            )

        arquivos = sorted(os.listdir(pasta))

        for arquivo in arquivos:
            if not arquivo.lower().endswith(
                EXTENSOES_IMAGEM
            ):
                continue

            caminho = os.path.join(pasta, arquivo)

            print(f"Analisando: {classe} - {arquivo}")

            try:
                dados = analisar_imagem(
                    caminho, classe
                )

                if dados is not None:
                    resultados.append(dados)
                else:
                    erros.append(caminho)

            except Exception as erro:
                print(
                    f"Erro ao analisar {arquivo}: {erro}"
                )
                erros.append(caminho)

    # 7. SALVAR CSV
    if not resultados:
        raise ValueError(
            "Nenhuma imagem foi analisada. "
            "Confira as pastas e os arquivos."
        )

    df = pd.DataFrame(resultados)

    df.to_csv(
        ARQUIVO_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    # 8. MOSTRAR RESUMO
    tempo = (time.time() - inicio) / 60

    print("\nAnalise concluida!")
    print("Arquivo CSV criado:", ARQUIVO_CSV)
    print("Total de imagens analisadas:", len(df))
    print("Total de colunas:", len(df.columns))

    print("\nQuantidade por classe:")
    print(df["classe"].value_counts())

    print("\nQuantidade de arquivos com erro:", len(erros))

    if erros:
        print("\nArquivos que precisam ser conferidos:")
        for caminho in erros:
            print("-", caminho)

    print("\nPrimeiras linhas do resultado:")
    print(df.head().to_string(index=False))

    quantidades = df["classe"].value_counts()

    if (
        len(df) == 300
        and all(
            quantidades.get(classe, 0) == 100
            for classe in [
                "rg_antigo", "rg_novo", "invalido"
            ]
        )
    ):
        print(
            "\nOK: foram analisadas 300 imagens, "
            "100 por classe."
        )
    else:
        print(
            "\nATENCAO: confira se as quantidades "
            "por classe estao corretas."
        )

    print(f"\nTempo total: {tempo:.2f} minutos")


# 9. EXECUTAR
executar_analise()

Analisando: rg_antigo - 00022188_in.jpg
Analisando: rg_antigo - 00022189_in.jpg
Analisando: rg_antigo - 00022190_in.jpg
Analisando: rg_antigo - 00022191_in.jpg
Analisando: rg_antigo - 00022192_in.jpg
Analisando: rg_antigo - 00022193_in.jpg
Analisando: rg_antigo - 00022194_in.jpg
Analisando: rg_antigo - 00022195_in.jpg
Analisando: rg_antigo - 00022196_in.jpg
Analisando: rg_antigo - 00022197_in.jpg
Analisando: rg_antigo - 00022198_in.jpg
Analisando: rg_antigo - 00022199_in.jpg
Analisando: rg_antigo - 00022200_in.jpg
Analisando: rg_antigo - 00022201_in.jpg
Analisando: rg_antigo - 00022202_in.jpg
Analisando: rg_antigo - 00022203_in.jpg
Analisando: rg_antigo - 00022204_in.jpg
Analisando: rg_antigo - 00022205_in.jpg
Analisando: rg_antigo - 00022206_in.jpg
Analisando: rg_antigo - 00022207_in.jpg
Analisando: rg_antigo - 00022208_in.jpg
Analisando: rg_antigo - 00022209_in.jpg
Analisando: rg_antigo - 00022210_in.jpg
Analisando: rg_antigo - 00022211_in.jpg
Analisando: rg_antigo - 00022212_in.jpg
